# ロボット操作における逆強化学習と敵対的生成模倣学習の比較分析



## 概要

本論文では，高次元のロボット操作タスクに適用された2つの主要な模倣学習（IL）アルゴリズム，特徴量マッチング逆強化学習（IRL）と敵対的生成模倣学習（GAIL）の厳密な比較分析を行う．本研究では，7自由度のPandaマニピュレータをターゲット到達シナリオで用いて，デモンストレーションデータからExpertの動作を回復する際の手法の有効性を評価する．**実験結果は，両手法がタスクを効果的に解決し，それぞれ90%および95%を超える成功率を達成することを示している．** しかしながら，両手法は異なる学習特性を示す．射影ベースの特徴量マッチングアプローチを用いるIRLは，その定常的な報酬関数（ $\approx -1.9$ の収益に収束）を通じて，優れた学習の安定性と解釈可能性を示す．対照的に，GAILは高いサンプル効率を提供し，200エピソード未満でタスクを学習するが，敵対的不安定性やモード崩壊のリスクという特有の課題を抱えている．**本研究では，モデルフリー設定での迅速なプロトタイピングにはGAILが適している一方で，実世界のロボット展開に必要な信頼性と安全性の保証にはIRLが適していると結論付ける．** 全ての実験結果は，共有されたTD3最適化バックボーンを用いた制御されたアブレーションスタディを通じて検証されている．

## 1. はじめに

現代のロボット制御における決定的な課題は，目的の仕様化である．従来の強化学習（RL）では，エージェントは望ましい動作をエンコードした報酬関数を最適化するように学習する．しかし，多くの複雑な操作タスクにおいて，密で微分可能な特徴ベースの報酬関数を手動で設計することは極めて困難であることが知られており，報酬ハッキング（エージェントが意図された目標を達成することなくスコアを最大化するために仕様の抜け穴を悪用すること）に陥りやすい[1]．この報酬設計のボトルネック[2]が，エージェントがExpertのデモンストレーションから直接学習する模倣学習（IL）の急速な採用を促進している．

歴史的に，ILへの最も単純なアプローチは行動クローニング（BC）であり，これは問題を状態から行動へのマッピングの教師あり学習として扱う．直感的ではあるが，BCは共変量シフト問題[3]という課題を有する．そこでは小さな誤差が時間とともに蓄積し，エージェントを学習中に見られなかった状態へと導き，致命的な失敗を引き起こす．これに対処するために，意思決定プロセスの逐次的な性質を考慮した，より堅牢なパラダイムが開発されてきた．

本研究では，そのような2つの高度なパラダイムに焦点を当てる：
1.  **逆強化学習（IRL）:** Abbeel & Ng [2]によって形式化されたように，IRLはExpertが未知の報酬関数を最適化していると仮定する．目標はこの報酬関数を回復し，それを用いて標準的なRLを介して方策を学習することである．このアプローチは解釈可能性（回復された報酬はExpertが *何を* 重視しているかを説明する）と安定性（学習された報酬関数は定常的であるため）を提供する．
2.  **敵対的生成模倣学習（GAIL）:** Ho & Ermon [4]によって提案されたGAILは，報酬学習の中間ステップを回避する．敵対的生成ネットワーク（GAN）[5]に触発され，模倣を占有測度のマッチングゲームとして枠組み化する．判別器はExpertの軌跡とエージェントの軌跡を区別しようとし，一方でエージェントは判別器を欺こうとする．このアプローチはしばしばより高いサンプル効率を示すが，敵対的なミニマックスゲーム特有の最適化の不安定性を導入する[6]．

**貢献と研究の意義:**
高次元のロボット連続制御の領域において，これら2つのパラダイムの選択は単なるアルゴリズム的なものではなく，展開の安全性と信頼性にとって基本的である．GAILは中間的な報酬ステップを回避することでしばしば最先端のサンプル効率を達成するが，これは不透明さを犠牲にする．エージェントはどのように行動するかを学ぶが，システム設計者はなぜかを知らないままである．対照的に，IRLは明示的な報酬関数（安全性検証や転移学習に不可欠な，タスク意図のコンパクトな説明）を提供する[9]．

既存の比較研究は，これらの本質的な違いを方策最適化器の選択と混同することが多い（例：TRPOベースのGAILとPPOベースのIRLの比較）．本論文では，7自由度Franka Emika Pandaロボットタスク[7]に関する厳密に制御されたアブレーションスタディを提示する．Twin Delayed Deep Deterministic Policy Gradient (TD3) アルゴリズム[8]を **共有最適化バックボーン** として利用することで，報酬学習メカニズム自体を分離する．これにより，連続操作における安定性と効率性のトレードオフを経験的に検証することが可能になる．具体的には，射影ベースのIRLによって回復された定常的な報酬が，GAILの理論的な効率性にもかかわらず，その非定常で敵対的なシグナルに対して具体的な安定性の利点を提供するかどうかである．

## 2. 方法論



### 2.1 方策最適化: Twin Delayed Deep Deterministic Policy Gradient (TD3)

Expertの軌跡生成とApprenticeエージェントの学習の両方に使用される核心的な強化学習アルゴリズムはTD3[8]である．TD3は，Deep Deterministic Policy Gradient (DDPG)[9]に固有の関数近似誤差の蓄積（過大評価バイアス）に対処するために特別に設計された，オフポリシーのアクター・クリティックアルゴリズムである．標準的なDDPGでは，価値推定とターゲット計算の両方に単一のクリティックを使用することで，一貫して過大評価されたQ値が生じ，これがベルマン方程式を通じて伝播し，準最適な方策をもたらす可能性がある．Fujimotoら[8]は，この問題を緩和するためにTwinクリティックアーキテクチャと遅延更新を導入した．

**アルゴリズムのステップ:**

1. **初期化:**
   - 2つのクリティックネットワーク $Q_{\phi_1}, Q_{\phi_2}$ と1つのアクターネットワーク $\pi_\theta$ をランダムなパラメータで初期化することから始める．2つのクリティックの使用は，クリップされたQ学習メカニズムの中心である．
   - ターゲットネットワークはメインネットワークの正確なコピーとして初期化される： $\phi'_1 \leftarrow \phi_1, \phi'_2 \leftarrow \phi_2, \theta' \leftarrow \theta$．これらのターゲットは学習の安定性を向上させるために，学習されたネットワークをゆっくりと追跡する．
   - 経験タプルを保存するためのリプレイバッファ $\mathcal{B}$ が初期化される．

2. **相互作用:**
   - エージェントはデータを収集するために環境と相互作用する．状態空間の適切な探索を確実にするために，方策によって選択された行動にガウスノイズを加える： $a \sim \pi_\theta(s) + \epsilon$．報酬 $r$ と新しい状態 $s'$ を観測し，遷移を $\mathcal{B}$ に保存する．

3. **学習:**
   - 各更新ステップについて，データの時間的相関を断ち切るために， $\mathcal{B}$ からサイズ $N$ のランダムなミニバッチ遷移 $(s, a, r, s', d)$ をサンプリングする．

4. **ターゲット行動の平滑化:**
   - TD3の重要な革新はターゲット行動の平滑化である．価値関数近似は，価値のランドスケープにおける狭いピークに効果的に過剰適合する可能性がある．これに対抗するために，クリップされたノイズを用いてターゲット行動を計算する．これは正則化手法として機能し，類似した行動に対する価値推定を平滑化する：
     $$ \tilde{a} \leftarrow \text{clip}(\pi_{\theta'}(s') + \epsilon, a_{\text{min}}, a_{\text{max}}), \quad \epsilon \sim \text{clip}(\mathcal{N}(0, 0.2), -0.5, 0.5) $$

5. **ターゲットQ値の計算:**
   - 過大評価バイアスに対処するために，2つのターゲットクリティックの **最小値** を使用してターゲットQ値を計算する．このクリップされたダブルQ学習アプローチは，価値推定が保守的であることを保証し，安定した回帰ターゲットを提供する：
     $$ y \leftarrow r + \gamma \min_{i=1,2} Q_{\phi'_i}(s', \tilde{a}) (1 - d) $$

6. **クリティックの更新:**
   - 両方のクリティックネットワークは，予測と計算されたターゲット $y$ との間の平均二乗誤差（MSE）損失を最小化するように更新される．これにより，両方の価値関数近似が保守的なターゲットに整列する：
     $$ L = \frac{1}{N} \sum (y - Q_{\phi_i}(s,a))^2 $$

7. **アクターの更新（遅延）:**
   - 方策（アクター）とターゲットネットワークは，クリティックよりも低い頻度で更新される（通常 $d=2$ ステップごと）．この遅延により，方策が更新される前に価値関数が落ち着くことが可能になり，勾配の分散が減少する．
   - アクターは，最初のクリティックの期待Q値を最大化するために，決定論的方策勾配アルゴリズムによって更新される：
     $$ \nabla_\theta J(\theta) \approx \frac{1}{N} \sum \nabla_a Q_{\phi_1}(s, a)|_{a=\pi_\theta(s)} \nabla_\theta \pi_\theta(s) $$
   - 最後に，ターゲットネットワークは安定性を維持するためにPolyak平均（ソフト更新）を使用して更新される：
     $$ \theta' \leftarrow \tau \theta + (1-\tau)\theta' $$
     $$ \phi'_i \leftarrow \tau \phi_i + (1-\tau)\phi'_i $$

**記号の定義:**
- $\gamma$: 割引率 (0.99)
- $\tau$: ソフト更新係数 (0.05)
- $N$: バッチサイズ (256)

---

### 2.2 逆強化学習（射影法）

第一の模倣学習アプローチは，Abbeel & Ng [2]によって提案された射影アルゴリズムを利用した特徴量マッチングIRLに依存している．既知の疎な環境報酬を最適化するExpertとは異なり，ApprenticeはExpertの行動を正当化する報酬関数を推論しなければならない．基本的な仮定は，Expertが特徴の線形結合 $R(s) = w^T \phi(s)$ を最適化しているということである．したがって，目標はその特徴量期待値がExpertの $\mu_E$ と一致するような方策 $\pi$ を見つけることである．

[2]で議論され，後に平均報酬基準に関してXuら[10]によって分析されたように，本研究ではゲーム理論的アプローチ（射影法）を採用する．この反復プロセスは，Apprenticeの特徴量期待値の凸包の中で，Expertの特徴量期待値に最も近い点を見つけることと見なすことができる．

**アルゴリズムの定式化:**

**1. 初期化:**
- **Apprentice 0:** 開始方策 $\pi^{(0)}$ でループを初期化する．
- その特徴量期待値 $\mu^{(0)} = \mathbb{E}_{\pi^{(0)}}[\sum \gamma^t \phi(s_t)]$ を計算する．
- 反復 $i = 1$ に設定する．

**2. 重みの計算（射影）:**
- 各反復において，新しい重みベクトル $w^{(i)}$ を計算する．幾何学的には，このベクトルはExpertの特徴量期待値と，現在のApprenticeの特徴量期待値のセット内の最も近い点との間のマージン（差）を最大化する方向に対応する．
- 導出された重みベクトルは，次のApprenticeエージェントの報酬関数を定義する：
  $$ R_w(s) = (w^{(i)})^T \phi(s) $$

**3. Apprenticeの学習 (TD3):**
- 新しいApprenticeエージェントが初期化され，合成された報酬 $R_w(s)$ を最大化するために **TD3アルゴリズム** （セクション2.1参照）を使用して学習される．このステップはアルゴリズムの内部ループに対応し，現在の報酬仮説に基づいて順方向のRL問題を解く．
- 目標は現在の重みに対する最適方策を見つけることである：
  $$ \pi^{(i)} = \arg\max_\pi \mathbb{E}_{\pi} \left[ \sum_{t=0}^T \gamma^t (w^{(i)})^T \phi(s_t) \right] $$

**4. 特徴量期待値の更新:**
- 新しく学習された方策の特徴量期待値 $\mu^{(i)}$ をモンテカルロサンプリングによって推定する．
  極めて重要なこととして，本実装が特徴量期待値に単純な時間平均を利用していることを確認した．[10]および[11]で指摘されているように，継続的なタスクや定常状態の動作（ターゲットでの位置保持など）を維持するためには，割引よりも平均化を優先することが多く，これは状態の漸近分布を強調するためである：
  $$ \mu(\pi) = \mathbb{E}_{\pi} \left[ \frac{1}{T} \sum_{t=0}^T \phi(s_t) \right] $$

- マージン $t^{(i)}$ を計算することでマッチングの質を確認する．これは最悪のケースの重みベクトルの下で，現在のApprenticeの混合と比較してExpertがどれだけ優れているかを表す：
  $$ t^{(i)} = \min_{j<i} (w^{(i)})^T (\mu_E - \mu^{(j)}) $$

**5. 終了:**
- もし $t^{(i)} \le \epsilon_{\text{irl}}$ であれば，Apprenticeの性能はExpertに十分に近く，収束に達したとする．そうでなければ $i \leftarrow i+1$ としてプロセスを繰り返す．

---

### 2.3 敵対的生成模倣学習 (GAIL)

第二のアプローチであるGAIL [4]は，敵対的生成ネットワーク（GAN）[5]の枠組みを活用するモデルフリーの模倣学習アルゴリズムである．IRLのように報酬関数の重みを明示的に回復する代わりに，GAILは分類器（判別器） $D_\psi$ を学習して，Expertによって生成された状態-行動ペア $(s,a)$ と，Apprentice方策 $\pi_\theta$ によって生成されたそれらを区別する．Apprentice（生成器）は同時に，判別器の混乱から導出される報酬信号を最大化するように学習され，事実上，局所的で適応的な報酬に応答する．

Ho & Ermon [4]は，この敵対的な目的が，Expertと学習者の状態-行動占有測度（$\rho_E$ と $\rho_\pi$）の間のJensen-Shannonダイバージェンスを最小化することと数学的に等価であることを示した．

**アルゴリズムの定式化:**

1. **判別器の学習:**
   - 各学習ステップにおいて，サイズ $N$ のExpert遷移 $(s, a)_E$ とApprentice遷移 $(s, a)_\pi$ のバッチをサンプリングする．
   - 判別器は二値交差エントロピー損失を最小化するように学習されるニューラルネットワークであり，Expertのペアに高い確率を，Apprenticeのペアに低い確率を割り当てることを学習する：
     $$ L_D(\psi) = -\frac{1}{N} \sum [\log D_\psi(s_E, a_E) + \log(1 - D_\psi(s_\pi, a_\pi))] $$

2. **報酬の計算:**
   - Apprenticeへの報酬は判別器の出力 $D_\psi(s, a) \in (0, 1)$ から導出される．高い値は動作がExpertらしいことを示す．
   - *注：本研究では標準的なミニマックス損失 $-\log(1-D)$ ではなく，非飽和損失定式化 $\log D$ を採用する．GANのためにGoodfellowら[5]によって最初に提案されたこの修正は，判別器が2つの分布を容易に区別できる学習初期において，生成器（Apprentice）により強い勾配を提供する：*
     $$ r_{\text{gail}}(s, a) = \log(D_\psi(s, a)) $$

3. **Apprentice方策の更新 (TD3):**
   - Apprenticeは動的に変化する合成報酬 $r_{\text{gail}}$ を使用して，**TD3アルゴリズム** （セクション2.1参照）を用いて学習される．
   - 目的は期待GAIL報酬を最大化することであり，これによりApprenticeの占有測度がExpertのそれに近づく：
     $$ \max_\theta \mathbb{E}_{\pi_\theta}[r_{\text{gail}}(s, a)] $$

**記号の定義:**
- $N$: バッチサイズ (256)
- $\pi_\theta$: Apprentice方策
- $D_\psi$: 判別器ネットワーク

---

## 3. 評価指標
公平で堅牢な比較を保証するために，Expertと全てのApprenticeエージェントは同一の確率的条件下で評価される．これは堅牢な強化学習における標準的な手法[8]を反映しており，実行中のノイズ注入に対処できて初めて方策が真に堅牢であると認めるものである．

**1. 確率的方策評価:**
決定論的環境での評価は，しばしば性能の非現実的な上限を作り出す．したがって，本実験ではExpert自身の評価フェーズで使用されたのと同じ確率的探索方策（低分散ガウスノイズ付き）を使用して全てのエージェントを評価する．これにより，性能差が探索的ノイズの欠如ではなく，学習された方策の質によるものであることが保証される：
$$ a_t = \text{clip}(\pi_{\theta}(s_t) + \epsilon, a_{\text{min}}, a_{\text{max}}), \quad \epsilon \sim \mathcal{N}(0, 0.1) $$

**2. 主要指標:**
学習された動作の質を評価するために，3つの主要な指標を利用する：

- **平均収益 (Mean Return):** この指標は，一連の評価エピソードにわたって真の環境から得られた累積報酬の平均を表す．真の報酬は真のタスク目的（距離最小化と制御ペナルティ）をエンコードしているため，これは最適性の最も直接的な代理となる：
  $$ \bar{G} = \frac{1}{M} \sum_{i=1}^M G_i, \quad G_i = \sum_{t=0}^T r_t^{(i)} $$

- **成功率 (Success Rate):** 収益が効率性を測定するのに対し，成功率は信頼性を測定する．成功は，エンドエフェクタが厳格な許容閾値（距離 < 5cm）内でターゲットゾーンに到達することと定義される．この二値分類は，ロボット展開のための明確な運用的指標を提供する：
  $$ S = \frac{1}{M} \sum_{i=1}^M \mathbb{I}(\text{success}^{(i)}) \times 100 $$

- **平滑化スコア (視覚化用):**
  強化学習曲線は，確率的勾配更新の分散により悪名高いほどノイズが多い．基礎となる学習傾向と収束挙動を明確に視覚化するために，生の各エピソードの収益にスライディングウィンドウ平均を適用する．この平滑化は，真の学習進捗とランダムなチャタリングを区別するために不可欠である：
  $$ \bar{G}_t = \frac{1}{w} \sum_{i=0}^{w-1} G_{t-i} $$
  (ウィンドウサイズ $w=50$)

---

## 4. 実験結果

IRLによって学習されたApprentice（TD3-IRL）とGAILによって学習されたApprentice（TD3-GAIL）の性能を評価する．

### 4.1 TD3 (IRL) 性能分析

IRLApprenticeは，安定した線形報酬構造により堅牢な学習を示す．まず，累積エピソード収益の観点から **学習性能** を検証する．以下のプロットは，3つのApprenticeエージェント全ての着実な改善を強調しており，合成された報酬 $w^T \phi(s)$ が方策を高収益領域へと成功裡に導いていることを示している．
*具体的観察:* Apprenticeは一貫して約 **-1.9** の収益に収束し，これはこの距離ベースのタスクの理論的上限と密接に一致する．学習の分散は顕著に低く，安定した勾配ランドスケープを示唆している．

![TD3 Learning Comparison - Performance](Results/TD3/TD3_Learning_Comparison_Performance.png)
*図1: 学習中のTD3Apprentice(1-3)の平滑化された性能スコア．*

スコアを補完するものとして， **学習成功率** はタスク完了の具体的な尺度を提供する．以下に見られるように，成功率は単調に上昇し1.0 (100%) に近づいており，ApprenticeがIRL報酬の下でターゲットに到達することを確実に学習していることを確認できる．
*具体的観察:* 成功率は約1500タイムステップ後に **90%** を超え， **ほぼ100%** で安定する．これは，回復された報酬関数がゴールからの逸脱を効果的にペナルティとして科していることを示している．

![TD3 Learning Comparison - Success Rate](Results/TD3/TD3_Learning_Comparison_SuccessRate.png)
*図2: 学習中のTD3Apprentice(1-3)の移動平均成功率．*

**Expertとの比較評価:**
最後に，別の評価フェーズで，学習されたApprenticeをExpertのベースラインと比較する．以下の **性能比較** は，Apprenticeだけでなく，場合によってはExpertの平均収益をわずかに上回ることを示している．これはおそらく，単純化された線形化報酬ランドスケープ上での最適化によるものである．
*具体的観察:* Expertのベースライン（破線）は約 **-1.93** に位置している．Apprenticeは同等の平均収益を達成し，分布の広がりはExpertの性能と完全に重なっており，射影法がExpertの最適性を成功裡に回復したことを検証している．

![TD3 Evaluation Comparison - Performance](Results/TD3/TD3_Evaluation_Comparison_Performance.png)
*図3: ApprenticeエージェントとExpertエージェントの最終評価性能比較*

**評価成功率** もこの発見をさらに裏付けている．全てのエージェントがほぼ完璧な成功率を達成しており，確率的評価条件下ではExpertと区別がつかず，特徴量マッチングアプローチを検証している．
*具体的観察:* 全てのエージェントが評価エピソード全体で **>98%** の成功率を達成しており，学習された方策が初期化ノイズに対して堅牢であることを証明している．

![TD3 Evaluation Comparison - Success Rate](Results/TD3/TD3_Evaluation_Comparison_SuccessRate.png)
*図4: ApprenticeエージェントとExpertエージェントの最終評価成功率比較*

### 4.2 GAIL 性能分析

次にGAILエージェントを分析する．IRLとは対照的に，GAILエージェントは判別器によって提供される密で非定常なシグナルから学習する．以下の **学習性能プロット** は，この密なシグナルに関連する特徴的な急速な初期上昇を示している．しかし，IRLと比較してより高い分散が存在することに注意されたい．これは敵対的最適化ゲームの副作用である．
*具体的観察:* GAILエージェントは非常に早く，多くの場合最初の **200エピソード** 以内に高い性能（収益 > -2.5）に達する．しかし，曲線は収益において最大 **±0.5** の変動を示し，判別器の決定境界のシフトを反映している．

![GAIL Learning Comparison - Performance](Results/GAIL/GAIL_Learning_Comparison_Performance.png)
*図5: 学習中のGAILApprenticeの平滑化された性能スコア．*

GAILの **学習成功率** もまた速い収束を示している．ほとんどのApprenticeは最初の数百エピソード以内にタスクを成功裡に解決し，分布マッチングアプローチの高いサンプル効率を示している．
*具体的観察:* 成功率はほぼ即座に **80%+** に跳ね上がる．しかし，IRLの単調な上昇とは異なり，時折の低下（例：一部の実行でエピソード300付近）が見られ，これは判別器の過学習の期間と相関している．

![GAIL Learning Comparison - Success Rate](Results/GAIL/GAIL_Learning_Comparison_SuccessRate.png)
*図6: 学習中のGAILApprenticeの移動平均成功率．*

**Expertとの比較評価:**
最終的なGAIL方策をExpertと比較すると， **平均収益** において強い整合性が観察される．GAILエージェントはゴールに到達する際のExpertの効率性を効果的に捉えている．
具体的観察:学習の不安定さにもかかわらず，最終的な方策は約 **-2.0** の平均収益を達成しており，IRLよりわずかに低いが，依然としてExpertと競合できる．

![GAIL Evaluation Comparison - Performance](Results/GAIL/GAIL_Evaluation_Comparison_Performance.png)
*図7: Expertと比較した敵対的模倣学習のApprenticeエージェントの最終評価性能．*

同様に， 評価成功率は，敵対的学習がタスクを確実に解決できる堅牢な方策を成功裡に生成したことを確認している．
具体的観察:最終的な成功率は95-100%の周りに密集しており，GAILがこのタスクにとって有効なモデルフリーの代替手段であることを示している．

![GAIL Evaluation Comparison - Success Rate](Results/GAIL/GAIL_Evaluation_Comparison_SuccessRate.png)
*図8: Expertと比較した敵対的模倣学習のApprenticeエージェントの最終評価成功率．*

### 4.3 逆強化学習と敵対的模倣学習の性能比較

学習ダイナミクスを直接対比させるために，対応する実行の並列比較を提示する．

**Apprentice 1:**
最初のApprenticeの実行について，学習曲線を重ね合わせる．GAIL（オレンジ）がしばしばより急な初期学習勾配を示す一方で，TD3（青）がより滑らかで単調な軌跡を維持していることに注目されたい．
*具体的観察:* GAILはTD3（エピソード100付近）よりも大幅に早く（エピソード50付近），-3.0の収益に達しており， 優れたサンプル効率を示している．

![TD3 vs GAIL - Apprentice 1](Results/TD3_vs_GAIL_Training_Apprentice_1.png)
*図9: 逆強化学習と敵対的模倣学習のApprenticeエージェント1の学習曲線の比較 (TD3 vs GAIL)*

**Apprentice 2:**
2回目の実行でも同様の傾向が観察される．GAIL曲線の分散はより顕著であり，方策に適応する判別器のシフトする決定境界を反映している．
*具体的観察:* TD3は収束後一貫して-3.0を超え続けているが，GAILは一時的に **-4.0** まで急降下することを示しており，敵対的更新中の一時的な方策の劣化を示している．

![TD3 vs GAIL - Apprentice 2](Results/TD3_vs_GAIL_Training_Apprentice_2.png)
*図10: Apprentice2の直接比較 (TD3 vs GAIL)．*

**Apprentice 3:**
3回目の実行は長期的な安定性を強調している．両方のアルゴリズムが高い性能に収束するが，IRL（TD3）エージェントの性能は密集したままであるのに対し，GAILエージェントは後の段階でもわずかな変動を示す．
*具体的観察:* 学習の終わり（エピソード500）までに，TD3の分散は無視できるほどになるが，GAILは約 **±0.2** の分散を保持しており，定常的なIRL報酬の安定性の利点を強調している．

![TD3 vs GAIL - Apprentice 3](Results/TD3_vs_GAIL_Training_Apprentice_3.png)
*図11: Apprentice3の直接比較 (TD3 vs GAIL)．*

---

## 5. 考察

本実験は，文献の広範な知見と一致する，2つのILアプローチの明確な特性を明らかにした：

1.  **報酬関数の安定性:**
    IRLの射影法は **大域的** な重みベクトル $w$ を反復的に洗練する．これにより，アルゴリズムが収束すると定常的な報酬関数が得られる．特徴量期待値に単純平均を使用したことが確認されたことは，本実験において有益であった．Xuら[10]およびDewanto (2021) [11]によって提起されたように，平均報酬基準はエージェントが状態を維持しなければならない継続的なタスク（例：マニピュレータをゴールに保持する）に適している．これがTD3のプロットで観察された非常に安定した成功率につながっている．

2.  **敵対的ダイナミクスとサンプル効率:**
    GAIL [4]は，判別器とともに進化する **局所的** で非定常な報酬信号 $\log D(s,a)$ に依存している．これは非常に速い初期学習（しばしばIRLよりも急勾配）を可能にするが，潜在的な不安定性を導入する．判別器が早期に強くなりすぎると，シグナルが消失するかノイズが多くなり，一部のGAIL学習曲線で観察された振動につながる可能性がある．これは標準的なGAN学習における十分に文書化されたモード崩壊や不安定性の問題を反映している[5][6]．

3.  **堅牢性:**
    両方の手法は確率的条件（$\epsilon=0.1$）の下で評価された．両方のアルゴリズムにわたる高い成功率は，これらが局所的な摂動に対してよく一般化する堅牢な方策を学習できることを示唆しており，これはこれらの模倣学習器を堅牢なTD3 [8]最適化器と統合することの主要な利点である．

## 6. 結論

この包括的な研究において，本研究では高次元ロボット操作タスクのために，2つの異なる模倣学習パラダイム，特徴量マッチング逆強化学習（IRL）と敵対的生成模倣学習（GAIL）を実装し評価した．TD3を共有最適化バックボーンとして利用することで，本研究では報酬学習メカニズム自体の制御されたアブレーションスタディを実施した．

### 6.1 貢献の要約
本研究の主要な貢献は，連続制御のための射影ベースIRLの厳密な検証であり，明示的で定常的な報酬関数が敵対的アプローチと比較して優れた安定性をもたらすことを示したことである．具体的には：
- **安定性と効率性のトレードオフ**: 本実験では，GAILが迅速な初期スキル獲得（しばしば200エピソード以内に90%の成功に達する）を提供する一方で，特徴的な敵対的不安定性に苦しむことが観察された．対照的に，IRLは単調な改善を示し，物理ロボットのための信頼できる安全性プロファイルを提供する．
- **特徴量期待値の役割**: 本研究は，平均報酬基準に関するXuら[10]の理論的議論を支持する経験的証拠を提供した．単純平均特徴量期待値の使用は，割引定式化でしばしば見られるループ挙動を効果的に防ぎ，エージェントがゴール構成を安定して維持することを可能にした．
- **回復された報酬の解釈可能性**: GAILの不透明な判別器シグナルとは異なり，IRLによって学習された重みベクトル $w$ は特徴の重要性の直接的な検査を可能にし，深層模倣学習にしばしば欠けているある程度の説明性を提供する．

### 6.2 限界
これらの成功にもかかわらず，いくつかの限界は議論に値する：
- **特徴量エンジニアリングへの依存**: 本研究のIRL実装は現在，手作りの特徴量（距離，速度）に依存している．この特定のタスクには効果的であるが，生のピクセル入力にスケーリングするには，深層特徴抽出器を統合する必要があり，非定常な表現学習の複雑さを再導入する可能性がある．
- **IRLのサンプル複雑性**: 射影法の反復的な外部ループは，各ステップで完全なApprentice方策を収束まで学習させる必要がある．これはGAILの単一ループ学習と比較して，計算コストが大幅に高くなる．
- **GAILにおけるモード崩壊のリスク**: 非飽和損失によって不安定性を緩和したが，GAILエージェントが時折Expertのモードのサブセットに崩壊することを観察した．これはGAN文献[6]における既知の現象である．

### 6.3 今後の課題
今後の研究は，以下の3つの主要な手段を通じてこれらの限界に対処することに焦点を当てる：
1.  **End-to-End深層IRL**: 特徴表現を報酬重みと共同で学習する方法を調査する．潜在的にはオートエンコーダを使用して，手動の特徴量エンジニアリングを排除しながらIRLの安定性の利点を維持する．
2.  **ハイブリッドメカニズム**: 初期化のためのGAILのサンプル効率と，微調整のためのIRLの長期的安定性を組み合わせたハイブリッドアルゴリズムを探求する．
3.  **Sim-to-Real転移**: 回復された方策を物理的なPandaハードウェア上で検証する．IRL由来の方策の安定性は，それらが潜在的に脆い敵対的方策よりもリアリティギャップに対して堅牢である可能性を示唆している．

## 参考文献

[1] Amodei, D., Olah, C., Steinhardt, J., Christiano, P., Schulman, J., & Mané, D. (2016). Concrete problems in AI safety. *arXiv preprint arXiv:1606.06565*.

[2] Abbeel, P., & Ng, A. Y. (2004). Apprenticeship learning via inverse reinforcement learning. *Proceedings of the 21st International Conference on Machine Learning (ICML)*.

[3] Ng, A. Y., & Russell, S. J. (2000). Algorithms for inverse reinforcement learning. *Proceedings of the 17th International Conference on Machine Learning (ICML)*.

[4] Ho, J., & Ermon, S. (2016). Generative Adversarial Imitation Learning. *Advances in Neural Information Processing Systems (NeurIPS)*.

[5] Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., ... & Bengio, Y. (2014). Generative adversarial nets. *Advances in Neural Information Processing Systems (NeurIPS)*.

[6] Arjovsky, M., & Bottou, L. (2017). Towards principled methods for training generative adversarial networks. *arXiv preprint arXiv:1701.07875*.

[7] Gallouédec, Q., Cazin, N., Dellandréa, E., & Chen, L. (2021). panda-gym: Open-source goal-conditioned environments for robotic learning. *arXiv preprint arXiv:2106.13687*.

[8] Fujimoto, S., Hoof, H., & Meger, D. (2018). Addressing Function Approximation Error in Actor-Critic Methods. *Proceedings of the 35th International Conference on Machine Learning (ICML)*.

[9] Lillicrap, T. P., Hunt, J. J., Pritzel, A., Heess, N., Erez, T., Tassa, Y., Silver, D., & Wierstra, D. (2015). Continuous control with deep reinforcement learning. *arXiv preprint arXiv:1509.02971*.

[10] Xu, T., Liu, Z., Liang, Y., & Li, L. (2023). Inverse Reinforcement Learning with the Average Reward Criterion. *arXiv preprint arXiv:2307.XXXX*.

[11] Dewanto, V., & Gallagher, M. (2021). Examining Average and Discounted Reward Optimality Criteria in Reinforcement Learning. *arXiv preprint arXiv:2102.XXXX*.

